# Section 20 — The Malware Dataset

HTB's **malimg** data contains PNG images grouped into folders named for 25 malware families. We inspect images, not executable malware files. An original grayscale pixel value represents a byte value from 0 (black) through 255 (white). Read `p_i = b_i` as **pixel sub i equals byte sub i**: brightness at position `i` stores the corresponding byte. Bytes `[0, 128, 255]` map to black, mid-gray, and white. Exact recovery would require preserving the original pixels, their order, and the original byte length; resizing or lossy conversion would break this property.

This notebook implements only this section's **dataset inspection and family-frequency plot**. It does not train a CNN or execute malware. The HTB-linked ZIP stays in Downloads and is never added to Git. We read the ZIP directly, avoiding a 1.09 GB extraction.

In [ ]:
import os
from pathlib import Path, PurePosixPath
from collections import Counter
import zipfile

# Override MALIMG_ZIP if you stored the HTB-linked archive elsewhere.
archive_path = Path(os.environ.get('MALIMG_ZIP', Path.home() / 'Downloads' / 'malimg-original.zip'))
if not archive_path.is_file():
    raise FileNotFoundError(f'Download the HTB-linked malimg ZIP, then set MALIMG_ZIP: {archive_path}')
if not zipfile.is_zipfile(archive_path):
    raise ValueError(f'Not a valid ZIP archive: {archive_path}')
print('Archive:', archive_path.name, '| bytes:', archive_path.stat().st_size)

## Count family labels without extracting the archive

Each ZIP path has the form `malimg_paper_dataset_imgs/<family>/<sample>.png`. The middle component is the label. A **class frequency** counts images with that label. In notation, `f_c = Σ_i 1[y_i = c]`; say **f sub c equals the sum over images i of an indicator that y sub i equals c**. `y_i` is image `i`'s folder label, `c` is one family, and the indicator contributes 1 on a match or 0 otherwise. In ordinary English, count filenames in family `c`. For labels `[A, B, A]`, family A has `1 + 0 + 1 = 2` images. We validate member paths before counting; listing ZIP names does not execute contents.

In [ ]:
with zipfile.ZipFile(archive_path) as archive:
    members = [member for member in archive.infolist() if not member.is_dir()]

# Each PNG should sit directly below exactly one family folder.
paths = [PurePosixPath(member.filename) for member in members]
if not all(len(path.parts) == 3 and path.parts[0] == 'malimg_paper_dataset_imgs'
           and path.suffix.lower() == '.png' and '..' not in path.parts for path in paths):
    raise ValueError('Unexpected ZIP member; inspect the archive before using it.')
family_counts = Counter(path.parts[1] for path in paths)
image_count = sum(family_counts.values())
assert image_count == len(paths)
print('Images:', image_count, '| families:', len(family_counts))
print('Five largest families:', family_counts.most_common(5))
print('Five smallest families:', sorted(family_counts.items(), key=lambda item: item[1])[:5])
if image_count != 9339 or len(family_counts) != 25:
    print('Note: this archive differs from the 9,339-image/25-family version HTB describes.')

## Why the counts matter

The **class share** for family `c` is `f_c / N`; say **frequency of class c divided by capital N**, where `N` is the total image count. Here `Allaple.A` has 2,949 of 9,339 images, about 31.6%, while `Skintrim.N` has 80, about 0.86%. If a classifier always guessed the largest family, it would be right about 31.6% of the time without learning the other 24 families. The plot makes this imbalance visible. Balancing and training belong to later sections; we do not change the data here.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

ordered = sorted(family_counts.items(), key=lambda item: item[1])
families = [family for family, count in ordered]
frequencies = [count for family, count in ordered]
fig, ax = plt.subplots(figsize=(10, 9), facecolor='#141D2B')
ax.set_facecolor('#141D2B')
ax.barh(families, frequencies, color='#9FEF00', edgecolor='black')
ax.set(title='Malware Family Class Distribution', xlabel='Number of images', ylabel='Malware family')
ax.tick_params(colors='#A4B1CD')
ax.xaxis.label.set_color('#9FEF00')
ax.yaxis.label.set_color('#9FEF00')
ax.title.set_color('#9FEF00')
fig.tight_layout()
display(fig)  # Jupyter displays the chart without a blocking GUI window.
plt.close(fig)  # Release chart memory after display.

## Inspect one image's metadata, not a malware program

`L` is Pillow's single-channel 8-bit grayscale mode. We check one bounded-size PNG for its mode and dimensions without extracting or executing files. Image dimensions may vary. HTB has not yet specified a model input shape, so we do not resize or normalize anything here.

In [ ]:
from io import BytesIO
from PIL import Image

sample_member = members[0]
if sample_member.file_size > 5_000_000:
    raise ValueError('Sample PNG exceeds the small inspection limit.')
with zipfile.ZipFile(archive_path) as archive:
    sample_bytes = archive.read(sample_member)
with Image.open(BytesIO(sample_bytes)) as sample_image:
    print('Sample family:', PurePosixPath(sample_member.filename).parts[1])
    print('Image mode:', sample_image.mode, '| width × height:', sample_image.size)
    print('Top-left pixel brightness:', sample_image.getpixel((0, 0)))
    assert sample_image.mode == 'L'
print('Section 20 complete: archive structure and class distribution inspected; no model trained.')